In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/processed/listings_enriched.csv", low_memory=False)
df["occupancy_rate"] = 1 - (df["availability_365"] / 365)
df = df[df["price"].between(100, 10000)].copy()

print(f"Listings for recommendation: {len(df):,}")
print(df[["name","neighbourhood_cleansed","room_type","price"]].head(3))

Listings for recommendation: 22,801
                              name neighbourhood_cleansed        room_type  \
0  Nice room with superb city view            Ratchathewi  Entire Home/Apt   
3       Beautiful waterfront house             Don Mueang  Entire Home/Apt   
4  Condo with Chaopraya River View             Rat Burana     Private Room   

    price  
0  1595.0  
3  4188.0  
4  1450.0  


In [ ]:
# --- Content-Based Recommendation System ---

# Build feature matrix for recommendations
rec_features = [
    "price", "accommodates", "bedrooms",
    "availability_365", "review_scores_rating",
    "neighbourhood_median_price", "occupancy_rate"
]

rec_df = df[["id", "name", "neighbourhood_cleansed", "room_type"] + rec_features].dropna().reset_index(drop=True)

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rec_df[rec_features])

# Compute similarity matrix (sample 3000 for memory)
sample_df = rec_df.sample(3000, random_state=42).reset_index(drop=True)
X_sample = scaler.transform(sample_df[rec_features])
similarity_matrix = cosine_similarity(X_sample)

print(f"Similarity matrix: {similarity_matrix.shape}")

def recommend(listing_idx, top_k=5):
    """Recommend similar listings based on features"""
    sim_scores = list(enumerate(similarity_matrix[listing_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_k+1]  # exclude self
    
    print(f"\nQuery listing:")
    q = sample_df.iloc[listing_idx]
    print(f"  {q['name']} | {q['room_type']} | ฿{q['price']:,.0f} | {q['neighbourhood_cleansed']}")
    
    print(f"\nTop {top_k} recommendations:")
    for idx, score in sim_scores:
        r = sample_df.iloc[idx]
        print(f"  [{score:.3f}] {r['name']} | {r['room_type']} | ฿{r['price']:,.0f} | {r['neighbourhood_cleansed']}")

# Test recommendations
recommend(0)
recommend(100)
recommend(500)

DuplicateError: Expected unique column names, got:
- 'price' 2 times